# DPO với ràng buộc đáp án

M4 dùng cùng M1 và 1.000 cặp với M3. Kiểm đầu vào, chạy gate rồi mới train.
Source và launch record phải được ghim; output lưu vào run mới.


In [ ]:
import hashlib
import json
import os
import re
import runpy
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = os.environ.get('VQA_PROJECT_ROOT')
LAUNCH_RECORD = 'source/answer_preservation_seed42.json'
LAUNCH_SHA256 = ''
EXPECTED_SEED = 42
RUN_GATE = False
RUN_TRAINING = False
ATTEMPT = {'m4': 'initial'}
RESUME = {'m4': None}
INSPECTED = False


## 1. Kho dự án và môi trường

Chạy lần lượt các ô setup. `PROJECT_DIR` chỉ cần điền khi Drive có nhiều kho phù hợp.
Launch record ghim source, cấu hình và dữ liệu của từng seed; không tự chọn bản mới nhất.


In [ ]:
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', timeout_ms=600000)
if PROJECT_DIR:
    ROOT = Path(PROJECT_DIR)
else:
    candidates = [Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')]
    candidates += list(Path('/content/drive/.shortcut-targets-by-id').glob('*/Vietnamese_VQA_NLE'))
    roots = {p.resolve() for p in candidates
             if (p / 'reports/m1_selection/frozen_checkpoint3683_v2/selection.json').is_file()}
    if len(roots) != 1:
        raise ValueError('Set PROJECT_DIR to the mounted project folder; no unique match found')
    ROOT = roots.pop()
if not ROOT.is_dir():
    raise FileNotFoundError(ROOT)
print('Project:', ROOT)


In [ ]:
INSPECTED = False
if not re.fullmatch(r'[0-9a-f]{64}', LAUNCH_SHA256):
    raise ValueError('Set the pinned launch SHA-256 from the delivery receipt')
LAUNCH_PATH = ROOT / LAUNCH_RECORD
if hashlib.sha256(LAUNCH_PATH.read_bytes()).hexdigest() != LAUNCH_SHA256:
    raise ValueError('Training launch changed')
launch = json.loads(LAUNCH_PATH.read_text('utf-8'))
if (launch.get('schema') != 'preference_training_launch_v1'
        or launch.get('scope') != 'answer_preservation_experiment'
        or launch.get('data_locked') is not True or launch.get('expected_pairs') != 1000
        or launch.get('training_seed', 42) != EXPECTED_SEED):
    raise ValueError('Require the locked 1,000-pair launch for the selected seed')
record = launch

bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != (
    '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40'
):
    raise ValueError('Bootstrap checksum mismatch')
release = record['source_release']
if not re.fullmatch(r'source_[A-Za-z0-9_]+', release):
    raise ValueError('Invalid source release')
REPO = Path(subprocess.check_output([
    sys.executable, str(bootstrap), '--archive',
    str(ROOT / 'source/releases' / f'{release}.zip'),
    '--sha256', record['source_sha256'], '--work-root', '/content/vqa_sources',
], text=True).strip())

run_command = runpy.run_path(str(REPO / 'src/vqa_explanation/utils/process.py'))['run_command']

def run(command):
    run_command([str(part) for part in command], cwd=REPO)

ENV_DIR = Path('/content/vqa_envs') / release
PYTHON = ENV_DIR / 'bin/python'
run([sys.executable, REPO / 'scripts/setup_preference_environment.py',
     '--environment-dir', ENV_DIR])
print('Source:', release)
print('Python:', PYTHON)
RUN_FOLDER = ROOT / launch['run_group']
print('Training seed:', EXPECTED_SEED)
print('Run folder:', RUN_FOLDER)


In [ ]:
INSPECTED = False
run([PYTHON, REPO / 'scripts/restore_preference_bundle.py',
     '--archive', ROOT / launch['preference_archive'],
     '--sha256', launch['preference_archive_sha256'],
     '--destination', ROOT / launch['preference_root']])

ENVIRONMENT = REPO / launch['environment']
paths_code = (
    'import json,sys; from pathlib import Path; '
    'from vqa_explanation.utils.environment import load_environment; '
    'print(json.dumps({k:str(v) for k,v in load_environment(Path(sys.argv[1])).items()}))'
)
child_env = {k: v for k, v in os.environ.items() if k not in {'PYTHONPATH', 'PYTHONHOME'}}
child_env['PYTHONNOUSERSITE'] = '1'
paths = json.loads(subprocess.check_output(
    [str(PYTHON), '-c', paths_code, str(ENVIRONMENT)], cwd=REPO, text=True, env=child_env,
))
bundle = ROOT / launch['dataset_bundle']
if hashlib.sha256((bundle / 'bundle.json').read_bytes()).hexdigest() != (
    launch['dataset_bundle_sha256']
):
    raise ValueError('Dataset bundle changed')
if launch.get('image_subset_archive'):
    run([PYTHON, REPO / 'scripts/stage_preference_data.py', 'restore',
         '--archive', ROOT / launch['image_subset_archive'],
         '--sha256', launch['image_subset_sha256'],
         '--destination', paths['image_root']])
else:
    raise ValueError('This launch requires the verified preference image subset')


## 2. Kiểm đầu vào

Kiểm cả hai nhánh trước khi dùng GPU. Gate chạy năm update riêng; train chính nạp lại M1, không học tiếp adapter gate.


In [ ]:
INSPECTED = False
sys.path.insert(0, str(REPO / 'src'))
from vqa_explanation.training.preference_launch import launch_command, stage_output  # noqa: E402
from vqa_explanation.utils.environment import project_root  # noqa: E402

if project_root().resolve() != REPO.resolve():
    raise ValueError('A different project package is loaded; use a fresh notebook kernel')
for arm in ('m4',):
    run(launch_command(REPO, LAUNCH_PATH, ROOT, arm, 'inspect', RUN_FOLDER,
                       python_executable=PYTHON))
INSPECTED = True


## 3. Gate và huấn luyện M4


In [ ]:
if not INSPECTED:
    raise RuntimeError('Run input inspection first')
if RUN_GATE:
    run(launch_command(REPO, LAUNCH_PATH, ROOT, 'm4', 'gate', RUN_FOLDER,
                       python_executable=PYTHON))
else:
    print('M4 gate not requested')


In [ ]:
if not INSPECTED:
    raise RuntimeError('Run input inspection first')
if RUN_TRAINING:
    run(launch_command(REPO, LAUNCH_PATH, ROOT, 'm4', 'train', RUN_FOLDER,
                       RESUME['m4'], ATTEMPT['m4'], python_executable=PYTHON))
else:
    print('M4 training not requested')


## 4. Kết quả đã lưu


In [ ]:
for arm in ('m4',):
    result_path = stage_output(RUN_FOLDER, arm, 'train', ATTEMPT[arm]) / 'training_result.json'
    if result_path.is_file():
        result = json.loads(result_path.read_text('utf-8'))
        print(arm.upper(), result['status'], 'steps=', result['global_step'],
              'epochs=', result['epoch'])
    else:
        print(arm.upper(), 'no final report for this attempt:', result_path)
